In [1]:
#Carga de datos
import pandas as pd
import json
import numpy as np
from comun.Server_PD import download_dataframe_minio

from collections import Counter
import re

#Graficas
import seaborn as sns
import matplotlib.pyplot as plt
import isodate

from itertools import combinations
import ast
import os
from pathlib import Path

import comun.analisisutils as utils
import comun.filter_and_divide_data as filter


c:\Trabajos\c2526-R1\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
print(os.getcwd())

c:\Trabajos\c2526-R1\src\analisis


# Carga de datos

In [3]:
#Arreglamos la ruta
root = Path().resolve().parents[1]  
os.chdir(root)
print(os.getcwd())
df = filter.download_latest_extraction_correct(0)
df_fil = filter.download_latest_extraction_correct(2)

C:\Trabajos\c2526-R1
Filtrando datos, con filtro de subtitulos
Numero de videos con duraciones atípicas: 3105
Numero de videos sin información textual: 675
Numero de videos con poca representación de generos: 1433
Guardamos aproximadamente  11 % de videos sin subtitulos para niños y adultos
La longitud antes de filtrar los subtitulos era  26398 y ahora es  14615
Tenemos  1509  videos de niños y  13106  videos de adultos
Hay  1361  videos de niños con subtítulos
Partiendo de 31355, se han eliminado 4957, resultando en: 26398 filas


# Transformamos y añadimos columnas

In [4]:
#Densidad de habla
df['Palabras_Por_Minuto'] = df.apply(utils.calcular_wpm, axis=1)
df['Palabras_Por_Minuto'] = round(df['Palabras_Por_Minuto'], 2)
df_fil['Palabras_Por_Minuto'] = df_fil.apply(utils.calcular_wpm, axis=1)
df_fil['Palabras_Por_Minuto'] = round(df_fil['Palabras_Por_Minuto'], 2)

# Creamos un diccionario con un df por cada genero

In [5]:
diccionario_generos = utils.division_generos(df)
for clave in diccionario_generos:
    print(clave, ":", len(diccionario_generos[clave]))
    df = diccionario_generos[clave]
    df_Adult, df_Kids = utils.division_edad(df)
    print(f'    Adultos: {len(df_Adult)}')
    print(f'    Kids: {len(df_Kids)}')
#Los 0 se deben a que youtube Kids y youtube Movies
print(f'-----------------------------')
diccionario_generos_fil = utils.division_generos(df_fil)
for clave in diccionario_generos:
    print(clave, ":", len(diccionario_generos_fil[clave]))
    df_fil = diccionario_generos_fil[clave]
    df_Adult, df_Kids = utils.division_edad(df_fil)
    print(f'    Adultos: {len(df_Adult)}')
    print(f'    Kids: {len(df_Kids)}')

Film & Animation : 2666
    Adultos: 634
    Kids: 2032
Autos & Vehicles : 340
    Adultos: 241
    Kids: 99
Music : 2632
    Adultos: 1108
    Kids: 1524
Pets & Animals : 489
    Adultos: 154
    Kids: 335
Sports : 1241
    Adultos: 298
    Kids: 943
Short Movies : 0
    Adultos: 0
    Kids: 0
Travel & Events : 429
    Adultos: 318
    Kids: 111
Gaming : 1610
    Adultos: 652
    Kids: 958
Videoblogging : 0
    Adultos: 0
    Kids: 0
People & Blogs : 3784
    Adultos: 2957
    Kids: 827
Comedy : 73
    Adultos: 48
    Kids: 25
Entertainment : 4470
    Adultos: 1328
    Kids: 3142
News & Politics : 3845
    Adultos: 3820
    Kids: 25
Howto & Style : 2049
    Adultos: 877
    Kids: 1172
Education : 6323
    Adultos: 1795
    Kids: 4528
Science & Technology : 1123
    Adultos: 778
    Kids: 345
Nonprofits & Activism : 281
    Adultos: 131
    Kids: 150
Movies : 0
    Adultos: 0
    Kids: 0
Anime/Animation : 0
    Adultos: 0
    Kids: 0
Action/Adventure : 0
    Adultos: 0
    Kids: 0
Clas

Vemos que hay muchos generos que no tienen ninguna aparicion. No los vamos a tener en cuenta para hacer las graficas.

In [ ]:
print("Cantidad de generos sin filtrar:", len(diccionario_generos))

diccionario_generos = dict(filter(lambda item: len(item[1]) > 0, diccionario_generos.items()))

print("Cantidad de generos filtrados:", len(diccionario_generos))


In [ ]:
#Frecuencias
generos_nombres = list(diccionario_generos.keys())
frecuencias = [len(df) for df in diccionario_generos.values()]

#Ordenamos
datos_ordenados = pd.Series(frecuencias, index=generos_nombres).sort_values(ascending=False)

#Grafica
plt.figure(figsize=(10, 8))
sns.barplot(x=datos_ordenados.values, y=datos_ordenados.index, palette='viridis')
#Ticks
max_frecuencia = datos_ordenados.max()
tus_ticks = [0, 250, 500]

paso_automatico = 200 if max_frecuencia < 2000 else 500
ticks_automaticos = list(range(0, int(max_frecuencia) + paso_automatico, paso_automatico))

ticks_finales = sorted(list(set(tus_ticks + ticks_automaticos)))
plt.xticks(ticks_finales)


plt.title("Distribución de Vídeos en los 15 Géneros Principales")
plt.xlabel("Cantidad de Vídeos")
plt.ylabel("Género")
plt.grid(axis='x', linestyle='--', alpha=0.4)

plt.show()

Nos quedamos con 15 géneros con videos.

El genero más frecuente es "Education" con diferencia. Le sigue "Entertaainment". Se podría decir que son las dos categorías más generales. Cualquier video se puede intentar clasificar como "Education" o "Entertaainment", por lo que tiene sentido que aparezcan con más frecuencia. Otros géneros bastante comunes son "News & Politics", "People & Blogs", "Film & Animation" entre otros.

 De esos 15 vamos a trabajar con los 8 mas frecuentes a la hora de analizar los datos.

In [ ]:
claves_ordenadas = sorted(diccionario_generos, key=lambda k: len(diccionario_generos[k]), reverse=True)

generos_seleccionados = claves_ordenadas[:8]

diccionario_top = {k: diccionario_generos[k] for k in generos_seleccionados}

print(f"Top 8 generos mas frecuentes: {list(diccionario_top.keys())}")

Filtramos también los df en funcion a si tienen subtitulos o no

In [ ]:
diccionario_subs = {
    clave: df[df['Subtitulos'].apply(lambda x: isinstance(x, str) and len(x.strip()) > 0)].copy() 
    for clave, df in diccionario_top.items()
}

for clave in diccionario_subs:
    print(f"{clave}: {len(diccionario_subs[clave])} vídeos con subtítulos.")

# Análisis no textual

## Duracion

In [ ]:
# Configuración
fig, ax = plt.subplots(4, 2, figsize=(16, 18), sharey=True)

ax = ax.flatten()
colores = ['darkseagreen', 'lightsalmon', 'salmon', 'thistle', 'khaki', 'darkkhaki', 'skyblue', 'plum']

for i, clave in zip(range(0,8), diccionario_top):
    utils.graficar_histograma_duracion(diccionario_top[clave], clave, ax[i], colores[i])

plt.tight_layout()
plt.show()

Como se puede observar los videos de "Music" y "Politics" tienden a ser más cortos. Concentrandose en duraciones de 5 minutos, a veces 10.
El resto se concentra en duraciones por debajo de 20 minutos, aunque si que tienen más variedad. 
Se puede apreciar como "Education" y "Entertainment" tienen más densidad de datos.

**COMPLETAR**

In [ ]:
df_grafica = pd.concat([
    df.assign(Genero=clave) for clave, df in diccionario_top.items()
])

plt.figure(figsize=(10, 5))
sns.boxplot(x='Genero', y='Duracion', data=df_grafica, palette='Set2', showfliers=False)  

plt.xticks(rotation=45)
plt.title('Comparativa de Duraciones')
plt.ylabel('Minutos')
plt.grid(axis = "y", alpha = 0.3)
plt.show()

En este gráfico podemos comprobar que sin ninguna duda, los videos más largos son los que se clasifican como "Film & Animation". Esto se debe a que la mayoría de videos bajo esta etiqueta serán largometrajes. Se aprecía también un amplio rango intercuartilico, lo que nos hace pensar que, además de peliculas, en esta categoría entran trailers u otro tipo de videos. 

Los videos musicales tienen una mediana muy baja, como era de esperar, pero nos sorprende que el bigote superior es el segundo más alto. Puede ser por videos con musica relajante para estudiar, videos de cine en los que se habla de la banda sonora u orquestas o conciertos.

Las cajas de "Entretainment" y "Gaming" son bastante similares. La diferencia más notable es que los videos de videojuegos suelen ser algo mas largos. Al ser "Entretainment" una categoría más amplia habrá más variabilidad entre los videos. "People & Blogs" también es bastante similar a estas dos.

Nos sorprende ver que "News and Politics" tiene una caja notablemente aplastada, con duraciones inferiores a 5 minutos. Esto puede que sea una estrategia para mantener la atención del espectador en videos complejos.

"Education" tiene una caja bastante estándar que nos hace ver que la mayoría de videos son muy consistentes.

## Densidad de habla

In [ ]:
df_grafica_subs = pd.concat([
    df.assign(Genero=clave) for clave, df in diccionario_subs.items()
])

plt.figure(figsize=(10, 5))

#No mostramos los outliers para que se vea más limpio
sns.boxplot(x='Genero', y='Palabras_Por_Minuto', data=df_grafica_subs, palette='Set2', showfliers=False)


plt.title('Densidad de Habla (WPM) en vídeos con Subtítulos')
plt.xlabel('Género')
plt.ylabel('Palabras por Minuto')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.show()

La caja que más llama la atención de esta grafica posiblement es la de "News & Politics". Se aprecia una densidad de habla notablemente superior al resto. Esto tiene sentido ya que suelen ser videos informativos que quieren hacer llegar un mensaje claro al espectador. Además, como hemos visto antes, suelen ser videos extremadamente cortos por lo que tienen que encajar una gran cantidad de datos en un corto periodo de tiempo.

"People & Blogs" y "Education" tienen tamaños bastante similares. Pero son géneros bastante independientes. No pensamos que haya una relación directa, sino más bien se puede tratar de una casualidad. Podemos ver que la mediana en "Education" es extremadamente baja, y no solo en esta sino también en el resto, excepto en "People & Blogs". 

También destaca que en "Film & Animation" la caja está completamente aplastada. 

Como podemos ver la densidad de habla se ve afectada por la duración de los videos. Cuanto mas cortos son los videos mayor densidad de habla hay. Por eso "News & Politics" tiene una densidad de habla tan alta y "Film & Animation" parece que no tiene.

# Analisis Textual

## Título

In [ ]:
diccionario_tf_titulos = {}

for clave, df_genero in diccionario_top.items():
    diccionario_tf_titulos[clave] = utils.tfidf_ngrams_titles(df_genero, "Titulo")


In [ ]:
common_terms = utils.common_terms_dictionary(diccionario_tf_titulos, "Titulo")

common_terms.sort_values("Score_Std", ascending=False).head(10)

La palabra "kids" aparece sobretodo en "Education" y "Entretaiment", con presencía en "Film & Animation". Rara vez aparece en "News & Politics". Todo esto lo hemos podido observar en el analisis de los videos infantiles.

Como era de esperar "Learn" tiene muchas apariciones en "Education".

La palabra "video" aparece equitativamente en todos los génros excepto en "Music" donde tiene mayor peso. Lo mismo pasa con "Official".

Ahora vamos a comparar las palabras que aparecen exclusivamente en un genero de entre dos que son pueden tener solapaciones.

In [ ]:
entertainment_only, gaming_only = utils.exclusive_terms(diccionario_tf_titulos["Entertainment"], 
    diccionario_tf_titulos["Gaming"], "Titulo")

print("Gaming")
print(gaming_only.head(5))
print("Entertainment")
print(entertainment_only.head(5))

Si observamos las palabras exculsivas entre dos generos que tienen cierta relación podemos sacar algunas palabras especificas de cada categoría.

Vemos que algunas palabras exclusivas de Gaming son; "walkthrough", "mod", "geometry dash" (un juego)...

Por otro lado en entretainment tenemos palabras como "cartoons", "songs", "blippi" (personaje infatil)... Estas palabras no son tan características. Seguramente se solapen con "Film & animation" o "Music".

In [ ]:
#Seleccionamos 4 generos
generos_a_comparar = ['Film & Animation', 'Music', 'Entertainment', 'Education']

plt.figure(figsize=(12, 8))
utils.comparativa_terminos(generos_a_comparar, diccionario_tf_titulos)
plt.show()


"kids" tiene el puntaje más alto de toda la gráfica dentro de Education.
Palabras como "learn", "english", "song" y "songs" muestran una relevancia masiva en esta categoría, lo que indica un fuerte enfoque en la enseñanza de idiomas y contenido infantil.

En la categoría "Music" como era de esperar, domina el término "music". También tiene una presencia fuerte en los términos "video" y "song", lo que sugiere que su TF-IDF está muy concentrado en etiquetas descriptivas del formato.

"Entertainment" muestra un perfil más equilibrado, pero destaca notablemente en el término "shorts" (superando a todas las demás categorías en esa palabra) y tiene una presencia sólida en "kids".

"Film & Animation" tiene puntuaciones generalmente más bajas y uniformes en comparación con las demás. Sus picos más altos están en "kids", "episode" y "cartoon", lo cual es lógico.


In [ ]:
#BERTOPIC SOLUCIONAR
topic_model_titulo, df_bertopic_titulo, temas_bertopic_titulo = utils.analizar_bertopic_dict(diccionario_top, "Titulo")


In [ ]:
dicc_bertopic_titulo = utils.division_generos(df_bertopic_titulo)
claves_ordenadas = sorted(dicc_bertopic_titulo, key=lambda k: len(dicc_bertopic_titulo[k]), reverse=True)

#Volvemos a filtrar el top 8
generos_seleccionados = claves_ordenadas[:8]
diccionario_top_bertopic_titulo = {k: dicc_bertopic_titulo[k] for k in generos_seleccionados}

for clave in diccionario_top_bertopic_titulo:
    print(f'Size of {clave, len(diccionario_top_bertopic_titulo[clave])}')


In [ ]:
generos_a_comparar = ["News & Politics", "People & Blogs", "Howto & Style", "Gaming"]

plt.figure(figsize=(10,6))
utils.graficar_bertopic_multiple(df_bertopic_titulo, generos_a_comparar, "Titulo")
plt.show()

In [ ]:
topic_info = topic_model_titulo.get_topic_info()
topic_info.head(10)

"News & Politics" domina los temas 4 (Guerra) y 8 (Clima).

"People & Blogs" domina el tema 0 (Historias personales/Dramas) y el 5 (Religión).

## Titulo canal


In [ ]:
diccionario_tf_titulo_canal = {}

for clave, df_genero in diccionario_top.items():
    diccionario_tf_titulo_canal[clave] = utils.tfidf_ngrams_titles(df_genero, "Titulo_canal")

**Terminos comunes**

In [ ]:
common_terms = utils.common_terms_dictionary(diccionario_tf_titulo_canal, "Titulo_canal")

common_terms.sort_values("Score_Std", ascending=False).head(20)

**Terminos exclusivos**

In [ ]:
#Entertainment vs Films & Animation
entertainment_only, films_only = utils.exclusive_terms(diccionario_tf_titulo_canal["Entertainment"], 
    diccionario_tf_titulo_canal["Film & Animation"], "Titulo_canal")

print("Film & Animation")
print(films_only.head(5))
print("Entertainment")
print(entertainment_only.head(5))

In [ ]:
#Seleccionamos 4 generos
generos_a_comparar = ["News & Politics", "People & Blogs", 'Entertainment', 'Education']

plt.figure(figsize=(12, 8))
utils.comparativa_terminos(generos_a_comparar, diccionario_tf_titulo_canal)
plt.show()

In [ ]:
topic_model_titulo_canal, df_bertopic_titulo_canal, temas_bertopic_titulo_canal = utils.analizar_bertopic_dict(diccionario_top, "Titulo_canal")

In [ ]:
dicc_bertopic_titulo_canal = utils.division_generos(df_bertopic_titulo_canal)
claves_ordenadas = sorted(dicc_bertopic_titulo_canal, key=lambda k: len(dicc_bertopic_titulo_canal[k]), reverse=True)

#Volvemos a filtrar el top 8
generos_seleccionados = claves_ordenadas[:8]
diccionario_top_bertopic_titulo_canal = {k: dicc_bertopic_titulo_canal[k] for k in generos_seleccionados}

for clave in diccionario_top_bertopic_titulo_canal:
    print(f'Size of {clave, len(diccionario_top_bertopic_titulo_canal[clave])}')


In [ ]:
generos_a_comparar = ['Film & Animation', 'Music', "Howto & Style", "Gaming"]

plt.figure(figsize=(10,6))
utils.graficar_bertopic_multiple(df_bertopic_titulo_canal, generos_a_comparar, "Titulo_canal")
plt.show()

In [ ]:
topic_info = topic_model_titulo_canal.get_topic_info()
topic_info.head(10)

## Tags


In [ ]:
diccionario_tf_tags = {}
diccionario_frec_tags = {}
for clave in diccionario_top:
    diccionario_tf_tags[clave] = utils.tfidf_tags(diccionario_top[clave]) 
    diccionario_frec_tags[clave] = utils.frecuencia_tags(diccionario_top[clave])


In [ ]:
common_terms = utils.common_terms_dictionary(diccionario_frec_tags, "Tag")
common_terms.sort_values("Frecuencia_Education", ascending=False).head(20)

Mirar que significa yt:cc = on

In [ ]:
common_terms = utils.common_terms_dictionary(diccionario_tf_tags, "Tag")

common_terms.sort_values("Score_Std", ascending=False).head(15)

In [ ]:
politics_only, education_only = utils.exclusive_terms(diccionario_frec_tags["News & Politics"], 
    diccionario_frec_tags["Education"], "Tag")

print("News & Politics")
print(politics_only.head(5))
print("Education")
print(education_only.head(5))

In [ ]:
generos_a_comparar = ['Film & Animation', 'Music', "Howto & Style", "Gaming"]

plt.figure(figsize=(10,6))
utils.graficar_bertopic_multiple(df_bertopic_titulo_canal, generos_a_comparar, "Titulo_canal")
plt.show()

## Descripción

In [ ]:

topic_model_descripcion, df_bertopic_descripcion, temas_bertopic_descripcion  = utils.analizar_bertopic_dict(diccionario_top, "Descripcion")

In [ ]:
dicc_bertopic_descripcion= utils.division_generos(df_bertopic_descripcion)
claves_ordenadas = sorted(dicc_bertopic_descripcion, key=lambda k: len(dicc_bertopic_descripcion[k]), reverse=True)

#Volvemos a filtrar el top 8
generos_seleccionados = claves_ordenadas[:8]
diccionario_top_bertopic_descripcion = {k: dicc_bertopic_descripcion[k] for k in generos_seleccionados}

for clave in diccionario_top_bertopic_descripcion:
    print(f'Size of {clave, len(diccionario_top_bertopic_descripcion[clave])}')


In [ ]:
generos_a_comparar = ['Film & Animation', "News & Politics", "People & Blogs"]

plt.figure(figsize=(10,6))
utils.graficar_bertopic_multiple(df_bertopic_descripcion, generos_a_comparar, "Descripcion")
plt.show()


## Subtítulos

In [ ]:

topic_model_subtitulos, df_bertopic_subtitulos, temas_bertopic_subtitulos  = utils.analizar_bertopic_dict(diccionario_top, "Descripcion")

In [ ]:
dicc_bertopic_subtitulos= utils.division_generos(df_bertopic_subtitulos)
claves_ordenadas = sorted(dicc_bertopic_subtitulos, key=lambda k: len(dicc_bertopic_subtitulos[k]), reverse=True)

#Volvemos a filtrar el top 8
generos_seleccionados = claves_ordenadas[:8]
diccionario_top_bertopic_subtitulos = {k: dicc_bertopic_subtitulos[k] for k in generos_seleccionados}

for clave in diccionario_top_bertopic_subtitulos:
    print(f'Size of {clave, len(diccionario_top_bertopic_subtitulos[clave])}')


In [ ]:
generos_a_comparar = ["News & Politics", "People & Blogs", 'Entertainment', 'Education']

plt.figure(figsize=(10,6))
utils.graficar_bertopic_multiple(df_bertopic_subtitulos, generos_a_comparar, "Subtitulos")
plt.show()